* Desc: Horror movie ratings vs. number of jump scares (top-N horror movies of all time)
* Rating: mean of IMDb, Letterboxd, Metacritic and Rotten Tomatoes critics' average,
  each scaled to 0-100; a movie needs at least 3 of the 4 scores
* Horror: at least one genre tag containing "horror" on IMDb, TMDB or Letterboxd
* Jump scares: https://www.whenjumpscare.com (major / minor), falling back to
  https://wheresthejump.com (not updated since 2023) when a movie is missing
* Data:
    * https://datasets.imdbws.com/ (`title.basics.tsv.gz`, `title.ratings.tsv.gz`)
    * `./data/TMDB_movie_dataset_v11.csv` (IMDb id <-> TMDB id, TMDB genres)
    * Wikidata (IMDb id -> Rotten Tomatoes / Metacritic ids)
    * Pages are cached in `./data/cache/horror/pages` by `horror_sources.fetch`

In [ ]:
import os
from datetime import date

import matplotlib.pyplot as plt
import polars as pl
from adjustText import adjust_text
from scipy.stats import spearmanr

import horror_sources as hs
from mapy.paths import DATA_DIR, OUTPUT_DIR

In [ ]:
IMDB_DIR = os.path.join(DATA_DIR, 'cache/horror/imdb')
TMDB_FILE = os.path.join(DATA_DIR, 'TMDB_movie_dataset_v11.csv')
MIN_IMDB_VOTES = 25000  # Popularity floor, keeps obscure movies with few votes out
MIN_IMDB_RATING = 6.0   # Loose floor, only to limit scraping; the cut-off below is far higher
MIN_SCORES = 3          # Out of 4 rating sources
N_TOP = 100
DISAGREEMENT = 0.30     # Report movies whose two jump scare counts differ by more than this

os.makedirs(OUTPUT_DIR, exist_ok=True)
for f in ['title.basics.tsv.gz', 'title.ratings.tsv.gz']:
    assert os.path.exists(os.path.join(IMDB_DIR, f)), \
        f'{f} not found! Get it from https://datasets.imdbws.com/{f}'

## Candidate pool: popular movies with a horror tag on IMDb or TMDB

In [ ]:
basics = pl.read_csv(os.path.join(IMDB_DIR, 'title.basics.tsv.gz'), separator='\t',
                     quote_char=None, null_values='\\N', infer_schema=False)
ratings = pl.read_csv(os.path.join(IMDB_DIR, 'title.ratings.tsv.gz'), separator='\t',
                      null_values='\\N', schema_overrides={'tconst': pl.Utf8})
tmdb = (pl.read_csv(TMDB_FILE, columns=['id', 'imdb_id', 'genres'], infer_schema=False)
        .drop_nulls('imdb_id').unique('imdb_id')
        .select(pl.col('imdb_id'), pl.col('id').cast(pl.Int64).alias('tmdb_id'),
                pl.col('genres').alias('tmdb_genres')))

df = (basics.filter(pl.col('titleType') == 'movie')
      .join(ratings, on='tconst')
      .filter((pl.col('numVotes') >= MIN_IMDB_VOTES)
              & (pl.col('averageRating') >= MIN_IMDB_RATING))
      .join(tmdb, left_on='tconst', right_on='imdb_id', how='left')
      .select(pl.col('tconst').alias('imdb_id'), 'tmdb_id',
              pl.col('primaryTitle').alias('title'),
              pl.col('startYear').cast(pl.Int64).alias('year'),
              pl.col('averageRating').alias('imdb_rating'),
              pl.col('numVotes').alias('imdb_votes'),
              pl.col('runtimeMinutes').cast(pl.Int64).alias('runtime'),
              pl.col('genres').alias('imdb_genres'),
              pl.col('tmdb_genres').fill_null('')))
is_horror = lambda c: pl.col(c).fill_null('').str.to_lowercase().str.contains('horror')
df = df.filter(is_horror('imdb_genres') | is_horror('tmdb_genres'))
print(f'{len(df)} horror candidates with >= {MIN_IMDB_VOTES} IMDb votes')

## Ratings: Letterboxd, Rotten Tomatoes, Metacritic

In [ ]:
# Parsers must reproduce known values for Alien (1979), checked by hand on 2026-10-08
alien = hs.letterboxd('tt0078748')
assert alien['lb_rating'] == 4.25 and 'Horror' in alien['lb_genres'], alien
assert alien['lb_tmdb_id'] == 348 and alien['lb_directors'] == {'scott'}
assert hs.rotten_tomatoes('m/alien')['rt_avg'] == 9.3
assert hs.rotten_tomatoes('m/nosferatu')['rt_year'] == 1922  # Not the 2008 re-release
assert hs.metacritic('movie/alien')['mc_score'] == 89

In [ ]:
wikidata = hs.wikidata_ids(df['imdb_id'].to_list())

rows, rejected = [], []
for imdb_id, title, year in df.select('imdb_id', 'title', 'year').iter_rows():
    ids = wikidata.get(imdb_id, {'rt': [], 'mc': []})
    lb = hs.letterboxd(imdb_id)
    directors = lb.get('lb_directors', set())
    # Same movie = release year within 1, or same director (re-releases, late US releases)
    rt, rt_rejected = hs.best_match([hs.rotten_tomatoes(x) for x in ids['rt']], year,
                                    directors, 'rt', 'rt_avg')
    mc, mc_rejected = hs.best_match([hs.metacritic(x) for x in ids['mc']], year,
                                    directors, 'mc', 'mc_score')
    rejected += [{'title': title, 'year': year, 'page': r.get('rt_url') or r.get('mc_url'),
                  'page_year': r.get('rt_year') or r.get('mc_year')}
                 for r in rt_rejected + mc_rejected]
    rows.append({'imdb_id': imdb_id, **lb, **rt, **mc})

scores = pl.DataFrame(
    [{k: v for k, v in r.items() if not k.endswith('_directors')} for r in rows],
    infer_schema_length=None).select(
    'imdb_id', 'lb_tmdb_id', 'lb_rating', 'lb_count', pl.col('lb_genres').list.join(', '),
    'rt_avg', 'rt_tomatometer', 'rt_reviews', 'rt_year', 'mc_score', 'mc_reviews', 'mc_year')
# The TMDB csv predates 2024, Letterboxd links every movie to its TMDB id
df = (df.join(scores, on='imdb_id', how='left')
      .with_columns(pl.coalesce('tmdb_id', 'lb_tmdb_id').alias('tmdb_id')))

print(f'Rejected {len(rejected)} pages of a different movie (remake, wrong id):')
print(pl.DataFrame(rejected))
print('Missing scores per source:')
df.select('lb_rating', 'rt_avg', 'mc_score').null_count()

## Horror check incl. Letterboxd, and borderline movies (horror on one source only)

In [ ]:
df = df.with_columns(
    (is_horror('imdb_genres').cast(pl.Int8) + is_horror('tmdb_genres').cast(pl.Int8)
     + is_horror('lb_genres').cast(pl.Int8)).alias('horror_tags'))

## Combined rating (0-100)

In [ ]:
scaled = ['imdb_100', 'lb_100', 'rt_100', 'mc_100']
df = df.with_columns(
    (pl.col('imdb_rating') * 10).alias('imdb_100'),
    (pl.col('lb_rating') * 20).alias('lb_100'),
    (pl.col('rt_avg') * 10).alias('rt_100'),
    pl.col('mc_score').cast(pl.Float64).alias('mc_100'),
).with_columns(
    pl.sum_horizontal(pl.col(c).is_not_null() for c in scaled).alias('n_scores'),
    pl.mean_horizontal(scaled).alias('rating'),
)
print(f'Dropped for < {MIN_SCORES} scores: {df.filter(pl.col("n_scores") < MIN_SCORES).height}')
print(df.filter(pl.col('n_scores') < MIN_SCORES).sort('imdb_votes', descending=True)
      .select('title', 'year', 'imdb_votes', *scaled).head(10))
df = df.filter(pl.col('n_scores') >= MIN_SCORES).sort('rating', descending=True)
df = df.with_columns(pl.int_range(1, pl.len() + 1).alias('rank'))

## Jump scares: WhenJumpScare (primary) and Where's The Jump (fallback, cross-check)

In [ ]:
wjs_urls = hs.wjs_movie_ids()
wjs = pl.DataFrame(
    [{'tmdb_id': t, **hs.whenjumpscare(u)} for t in df['tmdb_id'].drop_nulls()
     if (u := wjs_urls.get(t))],
    schema={'tmdb_id': pl.Int64, 'wjs_url': pl.Utf8, 'wjs_total': pl.Int64,
            'wjs_major': pl.Int64, 'wjs_minor': pl.Int64,
            'wjs_contributors': pl.List(pl.Utf8)})
print(f'WhenJumpScare: {wjs.height} candidates listed, '
      f'{wjs["wjs_total"].null_count()} pages without a parsable count')

wtj = pl.DataFrame(hs.wheresthejump()).with_columns(
    pl.col('wtj_title').map_elements(hs.normalize_title, return_dtype=pl.Utf8).alias('key'))
print(f"Where's The Jump: {wtj.height} movies, latest release year {wtj['wtj_year'].max()}")

df = df.join(wjs, on='tmdb_id', how='left').with_columns(
    pl.col('title').map_elements(hs.normalize_title, return_dtype=pl.Utf8).alias('key'))
# Title + year (+-1, release dates differ by country) join
wtj_match = (df.select('imdb_id', 'key', 'year').join(wtj, on='key')
             .filter((pl.col('wtj_year') - pl.col('year')).abs() <= 1)
             .unique('imdb_id', keep='first')
             .select('imdb_id', 'wtj_title', 'wtj_year', 'wtj_total', 'wtj_rating'))
df = df.join(wtj_match, on='imdb_id', how='left')

df = df.with_columns(
    pl.coalesce('wjs_total', 'wtj_total').alias('jump_scares'),
    pl.when(pl.col('wjs_total').is_not_null()).then(pl.lit('WhenJumpScare'))
      .when(pl.col('wtj_total').is_not_null()).then(pl.lit("Where's The Jump"))
      .alias('jump_source'),
)

## Top-N with jump scare data

In [ ]:
ranked = df.filter(pl.col('rank') <= df.filter(pl.col('jump_scares').is_not_null())
                   .get_column('rank').head(N_TOP).max())
top = ranked.filter(pl.col('jump_scares').is_not_null())
skipped = ranked.filter(pl.col('jump_scares').is_null())
print(f'Top {N_TOP} with jump scare data reaches down to rank {ranked["rank"].max()}, '
      f'combined rating >= {top["rating"].min():.1f}')
print(f'Skipped {skipped.height} higher-ranked movies without data on either site:')
print(skipped.select('rank', 'title', 'year', 'rating'))
print(top.group_by('jump_source').len())

In [ ]:
# Movies tagged horror by only one source - candidates to veto
print(top.filter(pl.col('horror_tags') == 1)
      .select('rank', 'title', 'year', 'imdb_genres', 'tmdb_genres', 'lb_genres'))

WhenJumpScare bulk-imported Where's The Jump: the imported timestamps are all credited to
one account (`bizu`) and match Where's The Jump exactly, including the major / minor split
(e.g. Halloween 1978: 13 scares, 2 major on both). So agreement between the two sites is not
an independent confirmation; only counts that other contributors edited or added are.

In [ ]:
IMPORTER = 'bizu'
top = top.with_columns(
    pl.col('wjs_contributors').list.eval(pl.element() != IMPORTER).list.any().fill_null(False)
      .alias('wjs_community'))
both = top.filter(pl.col('wjs_total').is_not_null() & pl.col('wtj_total').is_not_null())
diff = (pl.col('wjs_total') - pl.col('wtj_total')).abs() / pl.max_horizontal(
    'wjs_total', 'wtj_total').clip(lower_bound=1)
disagree = both.filter(diff > DISAGREEMENT)
print(f'{both.height} movies on both sites, {both.filter(diff == 0).height} identical counts')
print(f'{top["wjs_community"].sum()} of {top.height} movies have WhenJumpScare entries from '
      'contributors other than the importer')
print(f'{disagree.height} differ by more than {DISAGREEMENT:.0%}:')
print(disagree.select('rank', 'title', 'year', 'wjs_total', 'wjs_major', 'wtj_total',
                      'wjs_contributors'))

In [ ]:
pl.Config.set_tbl_rows(N_TOP)
print(top.select('rank', 'title', 'year', 'rating', 'imdb_100', 'lb_100', 'rt_100', 'mc_100',
                 'jump_scares', 'wjs_major', 'wtj_total'))

In [ ]:
top.select('rank', 'title', 'year', 'imdb_id', 'tmdb_id', 'rating', 'n_scores',
           'imdb_rating', 'imdb_votes', 'lb_rating', 'rt_avg', 'rt_tomatometer', 'mc_score',
           'jump_scares', 'jump_source', 'wjs_total', 'wjs_major', 'wjs_minor', 'wtj_total',
           'wjs_community', 'imdb_genres', 'tmdb_genres', 'lb_genres', 'horror_tags', 'wjs_url'
           ).write_csv(os.path.join(OUTPUT_DIR, 'horror_rating_vs_jump_scares.csv'))
# Every rated candidate with jump scare data, not only the top N (for robustness checks)
df.filter(pl.col('jump_scares').is_not_null()).select(
    'rank', 'title', 'year', 'imdb_id', 'rating', 'n_scores', 'imdb_votes', 'runtime',
    'jump_scares', 'jump_source', 'wjs_major', 'wtj_total'
    ).write_csv(os.path.join(OUTPUT_DIR, 'horror_rating_vs_jump_scares_all.csv'))

## 2-Axis Dot Plot: Rating vs Jump Scares

In [ ]:
SURFACE = '#fcfcfb'
TEXT_PRIMARY = '#0b0b0b'
TEXT_SECONDARY = '#52514e'
GRID = '#e4e3df'
NO_SPLIT = '#b4b2ab'    # No major / minor split: 0 scares, or Where's The Jump only
LEADER = '#8f8d86'
CMAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list(  # Sequential blue 250 -> 700
    'blues', ['#86b6ef', '#3987e5', '#256abf', '#104281', '#0d366b'])


def plot_rating_vs_jump_scares(data, n, figsize, fontsize):
    data = data.head(n)
    fig, ax = plt.subplots(figsize=figsize, facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    split = data.filter(pl.col('wjs_total') > 0)
    rest = data.filter(~(pl.col('wjs_total') > 0).fill_null(False))
    dots = ax.scatter(split['rating'], split['jump_scares'],
                      c=split['wjs_major'] / split['wjs_total'], cmap=CMAP, vmin=0, vmax=1,
                      s=70, edgecolors=SURFACE, linewidths=1.5, zorder=3)
    ax.scatter(rest['rating'], rest['jump_scares'], c=NO_SPLIT, s=70,
               edgecolors=SURFACE, linewidths=1.5, zorder=3)

    # Movies on the exact same point share one stacked label
    labels = (data.with_columns((pl.col('title') + ' (' + pl.col('year').cast(pl.Utf8) + ')')
                                .alias('label'))
              .group_by('rating', 'jump_scares', maintain_order=True)
              .agg(pl.col('label').str.join('\n')))
    texts = [ax.text(x, y, label, fontsize=fontsize, color=TEXT_PRIMARY)
             for x, y, label in labels.iter_rows()]
    adjust_text(texts, x=data['rating'].to_list(), y=data['jump_scares'].to_list(), ax=ax,
                expand=(1.2, 1.5), force_text=(0.3, 0.6), time_lim=5,
                arrowprops=dict(arrowstyle='-', color=LEADER, lw=0.6))

    rho, p = spearmanr(data['rating'], data['jump_scares'])
    ax.set_title(f'Top {n} horror movies: rating vs. jump scares', loc='left',
                 fontsize=18, color=TEXT_PRIMARY, pad=44)
    ax.text(0, 1.012, 'Highest rated horror movies with jump scare data. Spearman ρ = '
            f'{rho:.2f} (p = {p:.2f}).\nRating = mean of IMDb, Letterboxd, Metacritic and '
            'Rotten Tomatoes critics\' average, scaled to 0-100.',
            transform=ax.transAxes, fontsize=11, color=TEXT_SECONDARY, linespacing=1.5)
    ax.set_xlabel('Combined rating (0-100)', fontsize=12, color=TEXT_SECONDARY)
    ax.set_ylabel('Number of jump scares', fontsize=12, color=TEXT_SECONDARY)
    ax.grid(color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=TEXT_SECONDARY, labelsize=10)
    ax.set_ylim(bottom=-1)

    cbar = fig.colorbar(dots, ax=ax, fraction=0.025, pad=0.01)
    cbar.set_label('Share of major jump scares (WhenJumpScare)', color=TEXT_SECONDARY)
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(colors=TEXT_SECONDARY)
    if rest.height:
        ax.scatter([], [], c=NO_SPLIT, s=70,
                   label="No major/minor split (no scares, or Where's The Jump only)")
        ax.legend(loc='upper right', bbox_to_anchor=(1, -0.075), frameon=False,
                  labelcolor=TEXT_SECONDARY)
    fig.text(0.01, 0.005, 'Jump scares: whenjumpscare.com, wheresthejump.com. Ratings: '
             f'IMDb, Letterboxd, Metacritic, Rotten Tomatoes. Retrieved {date.today()}.',
             fontsize=9, color=TEXT_SECONDARY)

    output_file = os.path.join(OUTPUT_DIR, f'horror_rating_vs_jump_scares_top{n}.png')
    fig.savefig(output_file, dpi=150, bbox_inches='tight', facecolor=SURFACE)
    print(output_file, 'was saved.')
    plt.show()

In [ ]:
plot_rating_vs_jump_scares(top, 100, figsize=(24, 20), fontsize=8)

In [ ]:
plot_rating_vs_jump_scares(top, 50, figsize=(18, 12), fontsize=9)

In [ ]:
plot_rating_vs_jump_scares(top, 20, figsize=(14, 9), fontsize=11)